# Qwen3-8B smoke test

Starts Qwen3-8B-AWQ with vLLM on a T4, warms up with one short prompt, then times a 200-token generation. Temperature is 0 and thinking is off, matching the model client. The server is `http://127.0.0.1:8000/v1` inside this notebook.

In [ ]:
import subprocess

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
)
print(gpu.strip())
if "T4" not in gpu:
    raise RuntimeError("Set the notebook accelerator to GPU T4.")

In [ ]:
import subprocess

subprocess.check_call(["pip", "install", "-q", "vllm>=0.8.5"])

In [ ]:
import os
import subprocess
import time
import urllib.error
import urllib.request

log = open("/kaggle/working/vllm.log", "w")
env = os.environ.copy()
# T4 does not support the FlashInfer sampler.
env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
server = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen3-8B-AWQ",
        "--served-model-name", "Qwen/Qwen3-8B-AWQ",
        "--quantization", "awq",
        "--dtype", "float16",
        "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.90",
        "--port", "8000",
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
    env=env,
)

for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError("vLLM exited. See /kaggle/working/vllm.log")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2):
            break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(5)
else:
    raise RuntimeError("vLLM did not start. See /kaggle/working/vllm.log")

print("ready")

In [ ]:
import json
import time
import urllib.request

def complete(content, max_tokens):
    body = {
        "model": "Qwen/Qwen3-8B-AWQ",
        "messages": [{"role": "user", "content": content}],
        "temperature": 0,
        "max_tokens": max_tokens,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    request = urllib.request.Request(
        "http://127.0.0.1:8000/v1/chat/completions",
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
    )
    started = time.perf_counter()
    with urllib.request.urlopen(request, timeout=600) as response:
        payload = json.loads(response.read().decode())
    elapsed = time.perf_counter() - started
    usage = payload["usage"]
    return payload["choices"][0]["message"]["content"], usage["completion_tokens"], elapsed

# Warmup. Same prompt used for Flash and Flash-Lite.
text, tokens, seconds = complete("Reply with exactly: hello", 32)
print(text.strip(), tokens, f"{seconds:.2f}")

# Timed run of about 200 tokens, after warmup.
text, tokens, seconds = complete(
    "Count from 1 upward, one number per line. Do not stop early.", 200
)
print(tokens, f"{seconds:.2f}", f"{tokens / seconds:.2f}")